# Deep Dive: CUDA concept and programming
To be submitted in partial fulfillment of the requirements in Multiprocessing and Parallel Computing class (CEPARCO) using Cuda.

This notebook shall contain all the required variants for this project which is also specificed below.

---

## Project Specifications
Write the kernel in (1) C program; (2) an x86-64 assembly language; (3) x86 SIMD AVX2 assembly language using XMM register; (4) x86 SIMD AVX2 assembly language using YMM register. The kernel is to perform a **matrix vector product**.

**Input:** $m \times n$ Matrix $A$ (stored in row major order - $m$ rows and $n$ column) and vector $X$ of length $n$. Matrices and vectors are single-precision floating-point. You may assume square matrix ($m=n$).

**Process:** $y_i = \sum_{j=0}^{n-1} A_{i,j} \cdot x_j$

**Output:** store result in vector $Y$ of length $m$. Also, display the result of first 3 elements and last 3 elements of vector $Y$ for all versions of kernels.

---

## General Directions for this project
  1. This is a group project

  2. Your project specification is based on your group's "SIMD" project specifications

  3. For each specification, you will write multiple CUDA variants.  Separate notebook cell for each version.  Make sure to place a text header before the cell of each version.

  4. Variants:
    (1) a C/C++ program version (2) a CUDA program version using a grid-stride loop without prefetch and without mem advise (3) a CUDA program version using a grid-stride loop with prefetching but without page creation and without mem advise (4) a CUDA program version using a grid-stride loop with prefetch, with page creation but without mem advise (5) a CUDA program version using a grid-stride loop with prefetch, with page creation and with mem advise (6) Classic MemCopy method (no Unified memory) (7) ANother CUDA kernel that initializes the data

  5. For each kernel, execute at least 30 times. Observe the effect on the kernel execution time

  6. For the data, initialize each vector with values of your choice (please refer to your SIMD specifications for an example of data initialization.  Please document this value.

  7. Use nvprof to obtain execution time

  8. Check the correctness of your CUDA output vis-a-vis a "sanity check answer key".  This could be your C version. Output the correctness correspondingly.

  9. Place the result in GitHub (ensure I can access your GitHub).  The GitHub report should be inline and NOT linked to external files.

  10. The GitHub repository contains the following:
      1. Readme section containing the following report:
          * Group members, project specifications, AI usage declaration

          * screenshot of the program output with correctness check AND execution time for all cases

          * screenshot of nSight for all CUDA variants

          * comparative table of execution time (C, CUDA variants, x86-64, XMM, YMM) (see below some guide questions)

          * Analysis of results
            - Justify your kernel execution time.  
            - Analysis of speed performance across all platforms

          * Discuss the problems encountered and solutions made, unique methodology used, AHA moments, etc.

          * Discuss, based on your experience on the particular project use case, the differences between SIMD and SIMT in handling parallelism.  Include also the PROS and CONS of using SIMD and SIMT in your use case
      2. (TO BE CONTINUED)

---

### Variants Implemented
- [ ] Another CUDA kernel that initializes the data
- [x] CUDA program version using a grid-stride loop without prefetch and without mem advise NOTE : PA CHECK NALANG IT WORKED IN MY JUPYTER IDK WHY AYAW HERE HEHE - Chino
- [ ] CUDA program version using a grid-stride loop with prefetching but without page creation and without mem advise NOTE : I ADDED THIS NA DIN PADOUBLE CHECK NALANG - CHINO
- [ ] a CUDA program version using a grid-stride loop with prefetch, with page creation but without mem advise
- [ ] a CUDA program version using a grid-stride loop with prefetch, with page creation and with mem advise
- [x] Classic MemCopy method (no Unified memory) -Raidon
- [x] C/C++ - chino



---

---

# Environment Setup and Cuda Check

---


In [132]:
import os

# Add the directory containing the executable to the PATH
os.environ["PATH"] += os.pathsep + "/usr/local/cuda/bin"

# Check if the directory is added to the PATH
print(os.environ["PATH"])

/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda/bin:/usr/local/cuda/bin:/usr/local/cuda/bin


In [133]:
%%bash
nvcc --version
nvprof --version
nsys --version
ncu --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
nvprof: NVIDIA (R) Cuda command line profiler
Copyright (c) 2012 - 2024 NVIDIA Corporation
Release version 12.5.82 (21)
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2024 NVIDIA Corporation
Version 2024.2.1.0 (build 34372528) (public-release)


bash: line 3: nsys: command not found


In [106]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Tue Nov  4 18:23:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
---
# Data Initialization Kernel
---

In [107]:
%%writefile DataInit.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <curand_kernel.h>

__global__ void initializeData( float *A, float *X, int m, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    for (int i = idx; i < n; i += stride) {
        X[i] = sinf(i * 0.01f) * cosf(i * 0.007f) + 0.01f;
    }

    for (int i = idx; i < m * n; i += stride) {
        int row = i / n; // Calculate row
        int col = i % n; // Calculate col
        A[i] = 1.0f / ((row + 1.0f) * (col + 1.0f));
    }
}

Overwriting DataInit.cu


---
---
# Variant 1: CUDA program version using a grid-stride loop without prefetch and without mem advise
---

In [108]:
%%writefile CUDA_MATVEC_VAR1.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 10;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    // Initialize A and x
    for (size_t i = 0; i < m; i++){
        for (size_t j = 0; j < n; j++){
            A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
        }
    }
    for (size_t i = 0; i < n; i++)
        x[i] = cosf(i * 0.003f);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** function = MATVEC (float)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    // Multiple runs for nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    // Print first 3 and last 3 results (error check like SIMP spec requirement idk if still needed)
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    //Floating-point tolerant error check
    float tol = 1e-3f;
    size_t err_count = 0;

    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++){
            ref += A[i*n + j] * x[j];
        }
        if (fabsf(ref - y[i]) > tol)
            err_count++;
    }

    printf("Error count (CUDA program): %lu\n", (unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}


Overwriting CUDA_MATVEC_VAR1.cu


In [129]:
%%bash
# nvcc CUDA_MATVEC_VAR1.cu -o CUDA_MATVEC_VAR1 -Wno-deprecated-gpu-targets
nvcc -O3 -arch=sm_75 CUDA_MATVEC_VAR1.cu -o CUDA_MATVEC_VAR1 # for GPU (Tesla T4)

In [130]:
%%bash
nvprof ./CUDA_MATVEC_VAR1

*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==46658== NVPROF is profiling process 46658, command: ./CUDA_MATVEC_VAR1
==46658== Profiling application: ./CUDA_MATVEC_VAR1
==46658== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  107.69ms        10  10.769ms  8.8636ms  27.869ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   49.12%  108.82ms         3  36.272ms  6.4480us  108.75ms  cudaMallocManaged
                   48.60%  107.65ms         1  107.65ms  107.65ms  107.65ms  cudaDeviceSynchronize
                    2.08%  4.6171ms         3  1.5390ms  25.254us  4.5128ms  cudaFree
                    0.11%  249.93us        10  24.993us  5.0580us  196.47us  cudaLaunchKernel
                    0.07%  147.57us       114  1.2940us     106ns  54.372us  cuDeviceGetAttribute
                    0.01%  14.182us         1  14.182us  14.182us  14.182us  cuDeviceGetName
                    0.00%  10.306us         1  1

In [131]:
%%bash
nsys profile  -o CUDA_MATVEC_VAR1 ./CUDA_MATVEC_VAR1

bash: line 1: nsys: command not found


CalledProcessError: Command 'b'nsys profile  -o CUDA_MATVEC_VAR1 ./CUDA_MATVEC_VAR1\n'' returned non-zero exit status 127.

---
---
# Variant 2: CUDA program version using a grid-stride loop with prefetching but without page creation and without mem advise
---

In [111]:
%%writefile CUDA_MATVEC_VAR2.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// y = A * x (m x n matrix, row-major)
__global__
void matvec(unsigned long m, unsigned long n,
            const float* A, const float* x, float* y)
{
    unsigned long index  = blockIdx.x * blockDim.x + threadIdx.x;
    unsigned long stride = blockDim.x * gridDim.x;

    for (unsigned long i = index; i < m; i += stride) {
        float sum = 0.0f;
        unsigned long rowStart = i * n;
        for (unsigned long j = 0; j < n; ++j)
            sum += A[rowStart + j] * x[j];
        y[i] = sum;
    }
}

int main() {
  const unsigned long m = 4096;
  const unsigned long n = 4096;
  const size_t MATRIX_ELEMS = (size_t)m * n;
  const size_t MATRIX_BYTES = MATRIX_ELEMS * sizeof(float);
  const size_t X_BYTES = n * sizeof(float);
  const size_t Y_BYTES = m * sizeof(float);
  const size_t loope = 10;

  float *A, *x, *y;
  cudaMallocManaged(&A, MATRIX_BYTES);
  cudaMallocManaged(&x, X_BYTES);
  cudaMallocManaged(&y, Y_BYTES);

  int device;
  cudaGetDevice(&device);

  for (size_t i=0;i<MATRIX_ELEMS;i++)
      A[i] = sinf(0.0001f*i)*5.0f + 1.0f;
  for (size_t j=0;j<n;j++)
      x[j] = cosf(0.0003f*j)*3.0f + 0.5f;
  for (size_t i=0;i<m;i++)
      y[i] = 0.0f;

  cudaMemPrefetchAsync(A, MATRIX_BYTES, device, NULL);
  cudaMemPrefetchAsync(x, X_BYTES, device, NULL);
  cudaMemPrefetchAsync(y, Y_BYTES, device, NULL);

  size_t numThreads=1024;
  size_t numBlocks=(m + numThreads-1)/numThreads;

  printf("*** MATVEC VAR3: Prefetch only ***\n");
  printf("m=%lu, n=%lu\n",m,n);
  printf("Blocks=%zu, Threads=%zu\n",numBlocks,numThreads);

  for(size_t i=0;i<loope;i++)
    matvec<<<numBlocks,numThreads>>>(m,n,A,x,y);

  cudaDeviceSynchronize();

  cudaMemPrefetchAsync(y, Y_BYTES, cudaCpuDeviceId, NULL);
  cudaDeviceSynchronize();

  printf("y[0..2] = { %.3f, %.3f, %.3f }\n",y[0],y[1],y[2]);
  printf("y[-3..-1] = { %.3f, %.3f, %.3f }\n",y[m-3],y[m-2],y[m-1]);

size_t err = 0;
for (unsigned long i = 0; i < m; ++i) {
    double sum = 0.0;                    // high-precision reference
    size_t row = i * n;
    for (unsigned long j = 0; j < n; ++j)
        sum += (double)A[row + j] * (double)x[j];

    double got = (double)y[i];
    // relative tolerance scaled by result magnitude
    double rel_tol = 1e-3;               // ~1e-3 is reasonable for float with long sums
    double tol = rel_tol * fmax(1.0, fabs(sum));
    if (fabs(sum - got) > tol) err++;
}
printf("Error count: %lu\n", (unsigned long)err);
  cudaFree(A);
  cudaFree(x);
  cudaFree(y);
  return 0;
}


Overwriting CUDA_MATVEC_VAR2.cu


In [112]:
%%bash
#nvcc CUDA_MATVEC_VAR2.cu -o CUDA_MATVEC_VAR2 -Wno-deprecated-gpu-targets
nvcc -O3 -arch=sm_75 CUDA_MATVEC_VAR2.cu -o CUDA_MATVEC_VAR2 # for GPU (Tesla T4)

In [113]:
%%bash
nvprof ./CUDA_MATVEC_VAR2

*** MATVEC VAR3: Prefetch only ***
m=4096, n=4096
Blocks=4, Threads=1024
y[0..2] = { 21658.219, 43143.305, 59388.309 }
y[-3..-1] = { -34479.539, -17256.947, 4718.266 }
Error count: 0


==45241== NVPROF is profiling process 45241, command: ./CUDA_MATVEC_VAR2
==45241== Profiling application: ./CUDA_MATVEC_VAR2
==45241== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  67.607ms        10  6.7607ms  6.7513ms  6.7689ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   55.82%  99.290ms         3  33.097ms  6.4000us  99.238ms  cudaMallocManaged
                   37.98%  67.566ms         2  33.783ms  4.2520us  67.561ms  cudaDeviceSynchronize
                    2.55%  4.5334ms         3  1.5111ms  23.174us  4.4431ms  cudaFree
                    2.37%  4.2120ms        10  421.20us  5.6340us  4.1550ms  cudaLaunchKernel
                    1.19%  2.1236ms         4  530.89us  8.9440us  1.8864ms  cudaMemPrefetchAsync
                    0.08%  144.34us       114  1.2660us     125ns  54.525us  cuDeviceGetAttribute
                    0.01%  12.643us        

---
---
# Variant 3: CUDA program version using a grid-stride loop with prefetch, with page creation but without mem advise
---

In [114]:
%%writefile CUDA_MATVEC_VAR3.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 10;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    int device = -1;
    cudaGetDevice(&device);

    //prefetch data to create cpu page memory
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,cudaCpuDeviceId,NULL);

    //prefetch data to create gpu page memory
    cudaMemPrefetchAsync(y,MATRIX_BYTES,device,NULL);

    // Initialize A and x
    for (size_t i = 0; i < m; i++){
        for (size_t j = 0; j < n; j++){
            A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
        }
    }
    for (size_t i = 0; i < n; i++)
        x[i] = cosf(i * 0.003f);

    //Prefetch data from cpu-gpu
    cudaMemPrefetchAsync(A,MATRIX_BYTES,device,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,device,NULL);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** function = MATVEC (float)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    // Multiple runs for nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    //prefetch data from gpu-cpu
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(A,VECTOR_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);

    // Print first 3 and last 3 results (error check like SIMP spec requirement idk if still needed)
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    //Floating-point tolerant error check
    float tol = 1e-3f;
    size_t err_count = 0;

    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++){
            ref += A[i*n + j] * x[j];
        }
        if (fabsf(ref - y[i]) > tol)
            err_count++;
    }

    printf("Error count (CUDA program): %lu\n", (unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}


Overwriting CUDA_MATVEC_VAR3.cu


In [115]:
%%bash
#nvcc CUDA_MATVEC_VAR2.cu -o CUDA_MATVEC_VAR2 -Wno-deprecated-gpu-targets
nvcc -O3 -arch=sm_75 CUDA_MATVEC_VAR3.cu -o CUDA_MATVEC_VAR3 # for GPU (Tesla T4)

In [116]:
%%bash
nvprof ./CUDA_MATVEC_VAR3

*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==45298== NVPROF is profiling process 45298, command: ./CUDA_MATVEC_VAR3
==45298== Profiling application: ./CUDA_MATVEC_VAR3
==45298== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  66.623ms        10  6.6623ms  6.6463ms  6.7104ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   48.95%  93.084ms         3  31.028ms  6.7290us  93.020ms  cudaMallocManaged
                   35.02%  66.598ms         1  66.598ms  66.598ms  66.598ms  cudaDeviceSynchronize
                   11.97%  22.768ms         8  2.8460ms  1.3890us  14.921ms  cudaMemPrefetchAsync
                    2.12%  4.0373ms        10  403.73us  3.6960us  3.9996ms  cudaLaunchKernel
                    1.85%  3.5260ms         3  1.1753ms  25.012us  1.8584ms  cudaFree
                    0.07%  134.32us       114  1.1780us     108ns  54.317us  cuDeviceGetAttribute
                    0.01%  12.618us        

---
---
# Variant 4: CUDA program version using a grid-stride loop with prefetch, with page creation and with mem advise
---

In [117]:
%%writefile CUDA_MATVEC_VAR4.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);

    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 10;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    int device = -1;
    cudaGetDevice(&device);

    //memory advise
    cudaMemAdvise(A, MATRIX_BYTES, cudaMemAdviseSetReadMostly, device);
    cudaMemAdvise(x, VECTOR_BYTES, cudaMemAdviseSetReadMostly, device);

    cudaMemAdvise(y, VECTOR_BYTES, cudaMemAdviseSetReadMostly, device);

    cudaMemAdvise(A, MATRIX_BYTES, cudaMemAdviseSetPreferredLocation, cudaCpuDeviceId);
    cudaMemAdvise(x, VECTOR_BYTES, cudaMemAdviseSetPreferredLocation, cudaCpuDeviceId);
    cudaMemAdvise(y, VECTOR_BYTES, cudaMemAdviseSetPreferredLocation, cudaCpuDeviceId);


    //prefetch data to create cpu page memory
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,cudaCpuDeviceId,NULL);

    //prefetch data to create gpu page memory
    cudaMemPrefetchAsync(y,VECTOR_BYTES,device,NULL);

    //Initialize A and X
    for(size_t i = 0; i< m; i++) {
      for(size_t j = 0; j < n; j++) {
        A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
      }
    }

    for (size_t i =0; i < n; i++)
      x[i] = cosf(i * 0.003f);

    //prefetch data from cpu to gpu
    cudaMemPrefetchAsync(A,MATRIX_BYTES,device,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,device,NULL);

    // cuda kernel setup
    size_t numThreads = 1024;
    size_t numBlock = (m + numThreads - 1) / numThreads;

    printf("** function = MATVEC (float)\n");
    printf("m = %lu, n=%lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n", numBlock, numThreads);

    // run multiple times for testing/nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
      matvec<<<numBlock, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    //prefetch data from gpu to cpu page memory
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(y,VECTOR_BYTES,cudaCpuDeviceId,NULL);

    // Print first 3 and last 3 results
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    float tol = 1e-3f;
    size_t err_count = 0;

    for(size_t i = 0; i < m; i++) {
      float ref = 0.0f;
      for (size_t j = 0; j < n; j++){
        ref += A[i*n + j] * x[j];
      }
      if (fabs(ref - y[i]) > tol)
        err_count++;
    }

    printf("Error count (CUDA program): %lu\n",(unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}

Overwriting CUDA_MATVEC_VAR4.cu


In [118]:
%%bash
#nvcc CUDA_MATVEC_VAR2.cu -o CUDA_MATVEC_VAR2 -Wno-deprecated-gpu-targets
nvcc -O3 -arch=sm_75 CUDA_MATVEC_VAR4.cu -o CUDA_MATVEC_VAR4 # for GPU (Tesla T4)

In [119]:
%%bash
nvprof ./CUDA_MATVEC_VAR4

** function = MATVEC (float)
m = 4096, n=4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==45353== NVPROF is profiling process 45353, command: ./CUDA_MATVEC_VAR4
==45353== Profiling application: ./CUDA_MATVEC_VAR4
==45353== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  62.734ms        10  6.2734ms  6.2662ms  6.2816ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   53.67%  101.09ms         3  33.696ms  7.0620us  101.02ms  cudaMallocManaged
                   33.30%  62.717ms         1  62.717ms  62.717ms  62.717ms  cudaDeviceSynchronize
                   11.07%  20.854ms         8  2.6068ms  8.6720us  14.865ms  cudaMemPrefetchAsync
                    1.67%  3.1453ms         3  1.0484ms  43.168us  2.3361ms  cudaFree
                    0.20%  371.41us        10  37.141us  3.3170us  334.40us  cudaLaunchKernel
                    0.07%  128.27us       114  1.1250us     105ns  50.227us  cuDeviceGetAttribute
                    0.02%  28.767us        

---
---
# Variant 5: Classic MemCopy method (no Unified memory)
---

In [120]:
%%writefile CUDA_MATVEC_VAR5.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t X_BYTES = n * sizeof(float);
    const size_t Y_BYTES = m * sizeof(float);
    const size_t LOOPS = 30;

    // Host memory allocation
    float *h_A = (float*)malloc(MATRIX_BYTES);
    float *h_x = (float*)malloc(X_BYTES);
    float *h_y = (float*)malloc(Y_BYTES);

    // Initialize A and x on host
    for (size_t i = 0; i < m; i++)
        for (size_t j = 0; j < n; j++)
            h_A[i*n + j] = sinf(i * 0.002f + j * 0.001f);

    for (size_t i = 0; i < n; i++)
        h_x[i] = cosf(i * 0.003f);

    for (size_t i = 0; i < m; i++)
        h_y[i] = 0.0f;

    // Device memory allocation
    float *d_A, *d_x, *d_y;
    cudaMalloc(&d_A, MATRIX_BYTES);
    cudaMalloc(&d_x, X_BYTES);
    cudaMalloc(&d_y, Y_BYTES);

    // Copy data from host to device
    cudaMemcpy(d_A, h_A, MATRIX_BYTES, cudaMemcpyHostToDevice);
    cudaMemcpy(d_x, h_x, X_BYTES, cudaMemcpyHostToDevice);
    cudaMemset(d_y, 0, Y_BYTES);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** VARIANT 5: MATVEC with Classic MemCopy (No Unified Memory)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, d_A, d_x, d_y);

    cudaDeviceSynchronize();

    cudaMemcpy(h_y, d_y, Y_BYTES, cudaMemcpyDeviceToHost);

    printf("y[0] = %.8e, y[%lu] = %.8e, y[%lu] = %.8e\n",
       h_y[0], m-1, h_y[m-1], m/2, h_y[m/2]);


    size_t err_count = 0;
    float max_rel_err = 0.0f;
    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++)
            ref += h_A[i*n + j] * h_x[j];

        // Use relative error for better comparison
        float abs_err = fabsf(ref - h_y[i]);
        float rel_err = abs_err / fmaxf(1.0f, fabsf(ref));

        if (rel_err > 1e-4f)  // 0.01% relative tolerance
            err_count++;

        max_rel_err = fmaxf(max_rel_err, rel_err);
}

printf("Error count (Variant 5): %lu\n", (unsigned long)err_count);
printf("Max relative error: %.6e\n", max_rel_err);

    cudaFree(d_A);
    cudaFree(d_x);
    cudaFree(d_y);
    free(h_A);
    free(h_x);
    free(h_y);

    return 0;
}

Overwriting CUDA_MATVEC_VAR5.cu


In [121]:
%%bash
# nvcc -O3 CUDA_MATVEC_VAR5.cu -o CUDA_MATVEC_VAR5
nvcc -O3 -arch=sm_75 CUDA_MATVEC_VAR5.cu -o CUDA_MATVEC_VAR5 # for GPU (Tesla T4)

In [122]:
%%bash
nvprof ./CUDA_MATVEC_VAR5

*** VARIANT 5: MATVEC with Classic MemCopy (No Unified Memory)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0] = -1.10005165e+02, y[4095] = 1.85830673e+02, y[2048] = -6.56716614e+01
Error count (Variant 5): 0
Max relative error: 7.226525e-05


==45408== NVPROF is profiling process 45408, command: ./CUDA_MATVEC_VAR5
==45408== Profiling application: ./CUDA_MATVEC_VAR5
==45408== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   94.49%  266.02ms        30  8.8674ms  8.8538ms  8.8821ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
                    5.51%  15.502ms         2  7.7511ms  3.1360us  15.499ms  [CUDA memcpy HtoD]
                    0.00%  3.5190us         1  3.5190us  3.5190us  3.5190us  [CUDA memset]
                    0.00%  3.2320us         1  3.2320us  3.2320us  3.2320us  [CUDA memcpy DtoH]
      API calls:   68.55%  265.87ms         1  265.87ms  265.87ms  265.87ms  cudaDeviceSynchronize
                   26.85%  104.12ms         3  34.707ms  5.5530us  104.01ms  cudaMalloc
                    4.09%  15.860ms         3  5.2867ms  66.285us  15.702ms  cudaMemcpy
                    0.36%  1.3788ms         3  459.60us

---
---
# Variant : C/C++

In [123]:
%%writefile C_matvec_var1.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// *** C function version: y = A * x (row-major A, single precision)
void matvec(size_t m, size_t n, float *A, float *x, float *y)
{
    for (size_t i = 0; i < m; i++) {
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++) {
            sum += A[i * n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main(void)
{
    const size_t N = 256;          // m = n = 256
    const size_t M = N;
    const size_t A_BYTES = M * N * sizeof(float);
    const size_t X_BYTES = N * sizeof(float);
    const size_t Y_BYTES = M * sizeof(float);
    const size_t loope = 10;       // for averaging

    // declare arrays
    float *A = (float*)malloc(A_BYTES);
    float *x = (float*)malloc(X_BYTES);
    float *y = (float*)malloc(Y_BYTES);

    clock_t start, end;

    // init
    for (size_t i = 0; i < M; i++) {
        for (size_t j = 0; j < N; j++) {
            A[i * N + j] = (float)((i + j) % 100) * 0.01f;
        }
    }
    for (size_t j = 0; j < N; j++) {
        x[j] = (float)j * 0.001f + 1.0f;
    }

    // fill-in cache (warm-up)
    matvec(M, N, A, x, y);

    // time here
    double elapse = 0.0, time_taken = 0.0;
    for (size_t i = 0; i < loope; i++) {
        start = clock();
        matvec(M, N, A, x, y);
        end = clock();
        time_taken = ((double)(end - start)) * 1E3 / CLOCKS_PER_SEC;
        elapse += time_taken;
    }

    printf("Function (in C) average time for %lu loops is %f milliseconds to execute a matrix-vector with size %lux%lu \n",
           loope, elapse / loope, M, N);

    // show results: first 3 + last 3
    printf("y[0..2]   = %f %f %f\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = %f %f %f\n", y[M-3], y[M-2], y[M-1]);

    // error checking (compute reference again)
    size_t err_count = 0;
    for (size_t i = 0; i < M; i++) {
        float sum = 0.0f;
        for (size_t j = 0; j < N; j++) {
            sum += A[i * N + j] * x[j];
        }
        if (y[i] != sum)
            err_count++;
    }
    printf("Error count (C program): %lu\n", err_count);

    // Free memory
    free(A);
    free(x);
    free(y);
    return 0;
}


Writing C_matvec_var1.c


In [124]:
%%bash
gcc -O3 -Wall -Wextra C_matvec_var1.c -o C_matvec_var1 -lm

In [125]:
%%bash
./C_matvec_var1

Function (in C) average time for 10 loops is 0.076900 milliseconds to execute a matrix-vector with size 256x256 
y[0..2]   = 129.566803 130.155182 130.745575
y[-3..-1] = 152.249023 151.697403 151.148834
Error count (C program): 0
